In [134]:
import json
import requests
from pathlib import Path

import duckdb
import polyline
import pandas as pd

from geopy.distance import geodesic

from to_gpx import to_gpx

In [135]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
GRAPHHOPPER_BASE_URL = "http://localhost:8989"
GPS_ACCURACY = 50

In [136]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"
ground_truth_path = DATA_BASE_PATH / "ground_truth_route.parquet"
newson_krumm_route_network_path = DATA_BASE_PATH / "road_network.parquet"

In [137]:
ground_truth_df = duckdb.query(
    f"""
    WITH gt AS (
        SELECT edge_id, traversed, ROW_NUMBER() OVER () as row_num
        FROM '{ground_truth_path}'
    )
    SELECT gt.edge_id, traversed, linestring
    FROM gt
    LEFT JOIN '{newson_krumm_route_network_path}' nkr 
    ON gt.edge_id = nkr.edge_id
    ORDER BY gt.row_num
    """
).to_df()
ground_truth_df

,edge_id,traversed,linestring
0,884147800801,1,"LINESTRING(-122.109748721123 47.6673012971878,..."
1,884147800802,1,"LINESTRING(-122.105398178101 47.6675292849541,..."
2,884147800421,1,"LINESTRING(-122.102048099041 47.6676607131958,..."
3,884147800422,1,"LINESTRING(-122.1028393507 47.6681300997734, -..."
4,884147800423,1,"LINESTRING(-122.103689610958 47.6685512065887,..."
...,...,...,...
571,884147801154,1,"LINESTRING(-122.14302957058 47.6376816630363, ..."
572,884147800845,1,"LINESTRING(-122.14302957058 47.6378801465034, ..."
573,884147800842,1,"LINESTRING(-122.14291960001 47.6386311650276, ..."
574,884147800843,1,"LINESTRING(-122.142908871174 47.6404094696045,..."


In [138]:
route_network_df = duckdb.query(
    f"""
    SELECT edge_id, linestring 
    FROM '{newson_krumm_route_network_path}'
    """
).to_df()
route_network_df

,edge_id,linestring
0,883991900000,"LINESTRING(-122.732318937778 47.8899192810059,..."
1,883991900001,"LINESTRING(-122.71107852459 47.8776508569717, ..."
2,883991900002,"LINESTRING(-122.707419991493 47.8761515021324,..."
3,883991900003,"LINESTRING(-122.707419991493 47.8761515021324,..."
4,883991900004,"LINESTRING(-122.715329825878 47.8818699717522,..."
...,...,...
158162,884152400184,"LINESTRING(-121.777908504009 47.4525502324104,..."
158163,884152400185,"LINESTRING(-121.781320273876 47.4532207846642,..."
158164,884152400186,"LINESTRING(-121.784308254719 47.4577805399895,..."
158165,884152400187,"LINESTRING(-121.782489717007 47.4503803253174,..."


In [139]:
def parse_linestring(linestring_series: pd.Series) -> pd.Series:
    track_segs = linestring_series.str.replace(r"^LINESTRING\(|\)$", "", regex=True)
    track_segs = track_segs.str.replace(",", ";").replace(r"\s+", " ", regex=True)
    track_segs = track_segs.str.split(";")
    track_segs = track_segs.apply(
        lambda x: [tuple((lat, lon)) for (lon, lat) in (point.split() for point in x)]
    )
    return track_segs

In [140]:
ground_truth_df["track_segs"] = parse_linestring(ground_truth_df["linestring"])
route_network_df["track_segs"] = parse_linestring(route_network_df["linestring"])

In [141]:
gps_df = duckdb.query(f"SELECT * FROM '{gps_data_path}'").to_df()
gps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [142]:
gps_df.head()

,recorded_timestamp,lon,lat
0,2009-01-17 20:27:37,-122.107083,47.667483
1,2009-01-17 20:27:38,-122.107067,47.667500
2,2009-01-17 20:27:39,-122.107067,47.667500
3,2009-01-17 20:27:40,-122.107033,47.667517
4,2009-01-17 20:27:41,-122.106983,47.667533


In [143]:
points = gps_df[["lon", "lat", "recorded_timestamp"]].to_numpy()

gpx_path = to_gpx(points, DATA_BASE_PATH / "gps.gpx")

In [144]:
def request_map_matching(point_gpx_path: Path, output_path: Path) -> str:
    url = f"{GRAPHHOPPER_BASE_URL}/match?profile=car&gps_accuracy={GPS_ACCURACY}&type=json&locale=pt_BR&details=osm_way_id"
    headers = {
        "Content-Type": "application/gpx+xml",
    }

    with open(point_gpx_path, "rb") as f:
        body = f.read()

    req = requests.post(
        url,
        headers=headers,
        data=body,
    )

    req.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(req.content)

    print(f"Map matching result saved to {output_path}")

    res =  json.loads(req.content)["paths"][0]

    return {
        "points": res["points"],
        "edge_ids": res["details"]["osm_way_id"],
    }

In [145]:
graphhopper_result = request_map_matching(
    point_gpx_path=gpx_path, output_path=DATA_BASE_PATH / "map_matched.json"
)
graphhopper_result

Map matching result saved to /home/jose_edsouza/Documentos/Faculdade/TCC/repo/dataset/newson-krumm/data/map_matched.json


{'points': 'aa}aHf`hhV?oIEcDByLKk@KS}@bB_@x@MVe@hA_@fAa@nA_@rAq@|@a@`@g@\\gEvBe@^W`@Qj@Ih@?v@B`@D\\H\\P^XX\\VpH`El@d@f@n@hA~Bh@fB\\p@d@nAV~@h@`CHpAJrD?dIKlo@E~EKpG?pCDzBHdBDn@VnCJ~@L|@VzA\\`B`@~Ap@tBt@nBx@hBpAxBlBbCh@p@lA`ArA~@zPvHl@p@lFhCjBz@hAd@~BfAlBfA`M`Gt@^l@VXwEBWH}ANcCZgDPsCHqB?mAEmBYeF[sFa@mGSmDK}BOeCEkAGe@cBkEcAcCy@sBmAmCUc@u@eAm@cAa@eA_@gAY{@c@qAOc@c@oAl@u@j@o@bBiBdFyFvAuA~BwB|@k@`Ag@`Aa@dAYlAWjAQxBWbBWz@Mf@OAe@@SFQn@mATk@ZcAxAqKn@mE`@{CX{BX_BLi@ZgATo@Zm@j@w@^e@rAsA\\Sf@SdAW|AQxC_@nC_@jAKt@KXMb@a@zB_ELWlG_LnAwBdDuGnA_Dn@uAbA_DjDuI|DuIdCgFxAsCpA{BVa@`@m@hCyChBoBzBiC~AeBhE{EvB}BbCqBlAw@zAy@ZU\\QjBi@n@OlBUh@CxAAbADdAJb@HpBp@fAh@fBvApClBnAr@zB|@tCnAv@b@~AbAdBrAdB`Bz@~@vAhBp@~@z@tAv@zAv@jBjB|E`DdK|@~Al@bA`BrBjGpHbAzAt@xA`EdLbDtIvAjDj@tAr@pAt@zAlCjEz@jAvApAfGpEr@d@jAx@XV~@t@hAlArC`D~@tA`@f@|@l@p@T~@N^A~@Kb@M~Bi@bDeAlEoAdGyA|C{@v@]bDq@lF{A\\GhACxBTpDb@fK|@r@J`B~@vBhAfAf@n@Lp@@rCIjA?bCUtTkA|He@vCKzAK|@Ch@DpAZlBh@`HnBnCt@xJtCrB^jCXlBNX?p@GZIVMRSPUP[N]Jc@Ls@ZsBNk@Z{@\\q@l@aAr@y@hKyKvCy

In [146]:
grapphopper_trajectory = [res[-1] + 883000000000 for res in graphhopper_result["edge_ids"]]
grapphopper_trajectory

[884147800801,
 884147800802,
 884147800421,
 884147800422,
 884147800423,
 884147800805,
 884147800804,
 884147800806,
 884147800764,
 884147800776,
 884147800761,
 884147800762,
 884147800760,
 884147800774,
 884147801630,
 884147801640,
 884147801639,
 884147801055,
 884147801054,
 884147801048,
 884147801050,
 884147801049,
 884147801030,
 884147801032,
 884147801031,
 884147801040,
 884147801041,
 884147801033,
 884147801035,
 884147801038,
 884147801039,
 884147801034,
 884147801037,
 884147801042,
 884147801043,
 884147801044,
 884147801045,
 884147801179,
 884147801178,
 884147801176,
 884147801256,
 884147801260,
 884147801264,
 884147801263,
 884147801262,
 884147801261,
 884147801259,
 884147801253,
 884147801246,
 884147801245,
 884147801257,
 884147801258,
 884147801244,
 884147801248,
 884147801247,
 884147801251,
 884147801249,
 884147801254,
 884147801250,
 884147801255,
 884147801252,
 884148400028,
 884148400026,
 884148400027,
 884148400029,
 884148400704,
 884148400

In [147]:
ground_truth_trajectory = ground_truth_df["edge_id"].to_list()
ground_truth_trajectory

[884147800801,
 884147800802,
 884147800421,
 884147800422,
 884147800423,
 884147800805,
 884147800804,
 884147800806,
 884147800764,
 884147800776,
 884147800761,
 884147800762,
 884147800760,
 884147800774,
 884147801630,
 884147801640,
 884147801639,
 884147801055,
 884147801054,
 884147801048,
 884147801050,
 884147801049,
 884147801030,
 884147801032,
 884147801031,
 884147801040,
 884147801041,
 884147801033,
 884147801035,
 884147801038,
 884147801039,
 884147801034,
 884147801037,
 884147801042,
 884147801043,
 884147801044,
 884147801045,
 884147801179,
 884147801178,
 884147801176,
 884147801256,
 884147801260,
 884147801264,
 884147801263,
 884147801262,
 884147801261,
 884147801259,
 884147801253,
 884147801246,
 884147801245,
 884147801257,
 884147801258,
 884147801244,
 884147801248,
 884147801247,
 884147801251,
 884147801249,
 884147801254,
 884147801250,
 884147801255,
 884147801252,
 884148400028,
 884148400026,
 884148400027,
 884148400029,
 884148400704,
 884148400

In [148]:
len(grapphopper_trajectory), len(ground_truth_trajectory)

(584, 576)

In [149]:
def calculate_distance(segs_id: int, rn_df: pd.DataFrame = route_network_df) -> float:
    seg = rn_df[rn_df["edge_id"].isin([segs_id])]["track_segs"].values

    if len(seg) != 1:
        raise ValueError(f"Expected one segment for edge_id {segs_id}, got {len(seg)}")
    
    seg = seg[0]

    total_distance = 0.0
    for i in range(len(seg) - 1):
        total_distance += geodesic(seg[i], seg[i + 1]).meters
        
    return total_distance

In [150]:
added = [seg for seg in grapphopper_trajectory if seg not in ground_truth_trajectory]
removed = [seg for seg in ground_truth_trajectory if seg not in grapphopper_trajectory]

print(f"Added segments: {len(added)}")
print(f"Removed segments: {len(removed)}")

Added segments: 16
Removed segments: 8


In [151]:
ground_truth_distance = sum(
    calculate_distance(seg_id, route_network_df) for seg_id in ground_truth_trajectory
)

added_distance = sum(
    calculate_distance(seg_id, route_network_df) for seg_id in added
)
removed_distance = sum(
    calculate_distance(seg_id, route_network_df) for seg_id in removed
)

print(f"Ground truth distance: {ground_truth_distance:.2f} m")
print(f"Added distance: {added_distance:.2f} m")
print(f"Removed distance: {removed_distance:.2f} m")

Ground truth distance: 80241.64 m
Added distance: 882.14 m
Removed distance: 597.60 m


In [152]:
def calculate_accuracy(gt_dist: float, added_dist: float, removed_dist: float) -> float:
    if gt_dist == 0:
        return 0.0
    return 1- (added_dist + removed_dist) / gt_dist

In [153]:
accuracy = calculate_accuracy(
    ground_truth_distance, added_distance, removed_distance
)

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 98.16%
